# Notebook 17 — Yield Model Deployment Readiness

## 1. Objective

Assess whether the saved yield-prediction model selected in Notebook 16 is ready for integration into BharatCrop Intelligence and identify the remaining requirements before production release.

## 2. Scope

1. Verify the saved model artifact and its metadata.
2. Validate the model's input contract and feature ordering.
3. Test representative valid inputs, malformed inputs, missing values, and unseen categorical values.
4. Check prediction consistency, finite outputs, and domain plausibility without silently modifying predictions.
5. Evaluate inference latency and basic failure behavior.
6. Document deployment risks, safeguards, and unresolved operational requirements.
7. Generate a machine-readable readiness report and an evidence-based release verdict.

## 3. Existing Evidence

Notebook 16 performed model selection, chronological evaluation, rolling temporal validation, error analysis, and saved-artifact checks. Its existing results should be reused where relevant rather than reproduced.

The selected model is a HistGradientBoostingRegressor-based pipeline with these documented features:

- `Crop`
- `Season`
- `Crop_Category`
- `State`
- `District`
- `FE_YearStart`

The target is `Yield` as recorded in the source dataset. Its units and interpretation must not be assumed beyond the evidence already documented.

## 4. Validation Principles

- Do not retrain, retune, or silently replace the selected model.
- Do not repeat Notebook 16's model comparisons or temporal error analysis.
- Do not assume that successful predictions are necessarily agronomically plausible.
- Do not invent acceptable yield limits, latency thresholds, or operational guarantees.
- Record observed failures separately from risks that cannot be tested in this notebook.
- Treat unseen categories, invalid inputs, and out-of-range predictions as evidence to investigate, not as reasons to silently discard records.

## 5. Production Requirements Beyond This Notebook

A notebook cannot independently certify the complete deployed service. Production release may also require API integration tests, authentication and security checks, monitoring, logging, resource limits, dependency management, deployment configuration, rollback procedures, and defined service-level objectives.

## 6. Expected Deliverables

- Artifact and input-contract verification.
- Robustness and prediction-behavior test results.
- Inference performance measurements.
- Documented risks, safeguards, and outstanding requirements.
- A machine-readable readiness report.
- A final verdict: ready for integration, conditionally ready, or not ready, with supporting evidence.

## 7. Completion Criterion

Complete this notebook when every planned check has an explicit result, failures and limitations are documented, and the integration-readiness decision is justified by evidence.

In [9]:

from pathlib import Path
import json
import joblib

# Locate the project root from the current notebook directory.
current_dir = Path.cwd()

project_root = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (
            path / "models" / "yield_prediction"
            / "individual_crop_hgb_location.joblib"
        ).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "models/yield_prediction/individual_crop_hgb_location.joblib. "
        f"Current working directory: {current_dir}"
    )

model_dir = project_root / "models" / "yield_prediction"
model_path = model_dir / "individual_crop_hgb_location.joblib"
metadata_path = (
    model_dir / "individual_crop_hgb_location_metadata.json"
)

# Verify both artifacts exist before loading.
if not model_path.is_file():
    raise FileNotFoundError(f"Model artifact not found: {model_path}")

if not metadata_path.is_file():
    raise FileNotFoundError(f"Model metadata not found: {metadata_path}")

# Load artifacts without changing them.
with metadata_path.open("r", encoding="utf-8") as file:
    model_metadata = json.load(file)

yield_model = joblib.load(model_path)

# Report what was actually loaded.
print("PROJECT ROOT :", project_root)
print("MODEL PATH   :", model_path)
print("METADATA PATH:", metadata_path)
print("MODEL TYPE   :", type(yield_model).__name__)
print("METADATA KEYS:", list(model_metadata.keys()))
print("ARTIFACT SIZE:", f"{model_path.stat().st_size:,} bytes")
print("STATUS       : Both artifacts loaded successfully.")


PROJECT ROOT : C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
MODEL PATH   : C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\yield_prediction\individual_crop_hgb_location.joblib
METADATA PATH: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\yield_prediction\individual_crop_hgb_location_metadata.json
MODEL TYPE   : Pipeline
METADATA KEYS: ['model_name', 'project', 'candidate_scope', 'features', 'categorical_features', 'numeric_features', 'target', 'training_rows', 'training_year_start', 'training_year_end', 'holdout_rows', 'holdout_evaluated_during_final_fit', 'validation_windows', 'validation_metrics', 'limitations', 'saved_at_utc']
ARTIFACT SIZE: 786,593 bytes
STATUS       : Both artifacts loaded successfully.


In [10]:

import numpy as np

# Expected inference contract documented in Notebook 16.
expected_features = [
    "Crop",
    "Season",
    "Crop_Category",
    "State",
    "District",
    "FE_YearStart",
]

expected_target = "Yield"

# Read the actual saved metadata.
metadata_features = model_metadata.get("features")
metadata_target = model_metadata.get("target")

print("MODEL CONTRACT")
print("-" * 60)
print("Model name        :", model_metadata.get("model_name"))
print("Candidate scope   :", model_metadata.get("candidate_scope"))
print("Metadata features:", metadata_features)
print("Categorical       :", model_metadata.get("categorical_features"))
print("Numeric           :", model_metadata.get("numeric_features"))
print("Target            :", metadata_target)
print("Training rows     :", model_metadata.get("training_rows"))
print("Training years    :", (
    model_metadata.get("training_year_start"),
    model_metadata.get("training_year_end"),
))
print("Holdout rows      :", model_metadata.get("holdout_rows"))

# Inspect the actual pipeline interface.
print("\nPIPELINE CONFIGURATION")
print("-" * 60)
print("Pipeline steps:", list(yield_model.named_steps.keys()))

if hasattr(yield_model, "feature_names_in_"):
    pipeline_features = list(yield_model.feature_names_in_)
    print("Pipeline input features:", pipeline_features)
else:
    pipeline_features = None
    print(
        "Pipeline input feature names are not exposed by the "
        "top-level estimator; inspect the preprocessing step next."
    )

# Validate what can be established from the saved artifacts.
checks = {
    "Metadata features match expected contract":
        metadata_features == expected_features,
    "Metadata target is Yield":
        metadata_target == expected_target,
    "Training row count matches Notebook 16":
        model_metadata.get("training_rows") == 35784,
    "Holdout row count matches Notebook 16":
        model_metadata.get("holdout_rows") == 7369,
    "Pipeline exposes expected feature order":
        pipeline_features is None or pipeline_features == expected_features,
}

print("\nCONTRACT CHECKS")
print("-" * 60)
for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'} | {description}")

if not all(checks.values()):
    print(
        "\nReview the failed checks before proceeding. "
        "Do not change metadata or model artifacts to force a pass."
    )
else:
    print("\nAll initial contract checks passed.")


MODEL CONTRACT
------------------------------------------------------------
Model name        : HistGradientBoostingRegressor
Candidate scope   : Individual Crop Candidate cohort
Metadata features: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Categorical       : ['Crop', 'Season', 'Crop_Category', 'State', 'District']
Numeric           : ['FE_YearStart']
Target            : Yield
Training rows     : 35784
Training years    : (2013, 2022)
Holdout rows      : 7369

PIPELINE CONFIGURATION
------------------------------------------------------------
Pipeline steps: ['preprocessor', 'model']
Pipeline input features: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']

CONTRACT CHECKS
------------------------------------------------------------
PASS | Metadata features match expected contract
PASS | Metadata target is Yield
PASS | Training row count matches Notebook 16
PASS | Holdout row count matches Notebook 16
PASS | Pipeline exposes expect

In [11]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocessor = yield_model.named_steps["preprocessor"]
estimator = yield_model.named_steps["model"]

print("PREPROCESSOR TYPE:", type(preprocessor).__name__)
print("ESTIMATOR TYPE   :", type(estimator).__name__)

print("\nPREPROCESSOR CONFIGURATION")
print("-" * 60)

if hasattr(preprocessor, "transformers_"):
    for name, transformer, columns in preprocessor.transformers_:
        print(f"\nTransformer: {name}")
        print("Columns     :", list(columns) if not isinstance(columns, str) else columns)
        print("Type        :", type(transformer).__name__)

        if hasattr(transformer, "get_params"):
            params = transformer.get_params(deep=False)
            for key in (
                "handle_unknown",
                "handle_missing",
                "strategy",
                "sparse_output",
                "drop",
            ):
                if key in params:
                    print(f"{key:16}:", params[key])

        # Inspect nested pipeline steps when present.
        if isinstance(transformer, Pipeline):
            for step_name, step in transformer.steps:
                print(f"  Step {step_name}: {type(step).__name__}")
                if hasattr(step, "get_params"):
                    step_params = step.get_params(deep=False)
                    for key in ("handle_unknown", "handle_missing", "strategy"):
                        if key in step_params:
                            print(f"    {key}: {step_params[key]}")
else:
    print("Fitted transformer details are not exposed in the expected format.")

print("\nESTIMATOR CONFIGURATION")
print("-" * 60)

for key, value in estimator.get_params(deep=False).items():
    print(f"{key}: {value}")

print("\nMETADATA LIMITATIONS")
print("-" * 60)
print(model_metadata.get("limitations", "No limitations field found."))


PREPROCESSOR TYPE: ColumnTransformer
ESTIMATOR TYPE   : HistGradientBoostingRegressor

PREPROCESSOR CONFIGURATION
------------------------------------------------------------

Transformer: categorical
Columns     : ['Crop', 'Season', 'Crop_Category', 'State', 'District']
Type        : OneHotEncoder
handle_unknown  : ignore
sparse_output   : False
drop            : None

Transformer: numeric
Columns     : ['FE_YearStart']
Type        : FunctionTransformer

ESTIMATOR CONFIGURATION
------------------------------------------------------------
categorical_features: from_dtype
early_stopping: auto
interaction_cst: None
l2_regularization: 1.0
learning_rate: 0.08
loss: squared_error
max_bins: 255
max_depth: None
max_features: 1.0
max_iter: 200
max_leaf_nodes: 31
min_samples_leaf: 20
monotonic_cst: None
n_iter_no_change: 10
quantile: None
random_state: 42
scoring: loss
tol: 1e-07
validation_fraction: 0.1
verbose: 0
warm_start: False

METADATA LIMITATIONS
----------------------------------------

In [12]:

import pandas as pd
import numpy as np
import time

# Representative inputs for inference-contract testing.
# These are synthetic test cases, not claims about real farm conditions.
test_inputs = pd.DataFrame([
    {
        "Crop": "Rice",
        "Season": "Kharif",
        "Crop_Category": "Individual Crop Candidate",
        "State": "Punjab",
        "District": "Ludhiana",
        "FE_YearStart": 2023,
    },
    {
        "Crop": "Wheat",
        "Season": "Rabi",
        "Crop_Category": "Individual Crop Candidate",
        "State": "Punjab",
        "District": "Ludhiana",
        "FE_YearStart": 2024,
    },
], columns=expected_features)

test_cases = {
    "valid_inputs": test_inputs,
    "unseen_categories": test_inputs.copy(),
    "missing_crop": test_inputs.copy(),
    "missing_year": test_inputs.copy(),
    "invalid_year_string": test_inputs.copy(),
}

test_cases["unseen_categories"].loc[0, "Crop"] = (
    "__UNSEEN_CROP_TEST__"
)
test_cases["unseen_categories"].loc[0, "District"] = (
    "__UNSEEN_DISTRICT_TEST__"
)
test_cases["missing_crop"].loc[0, "Crop"] = np.nan
test_cases["missing_year"].loc[0, "FE_YearStart"] = np.nan
test_cases["invalid_year_string"].loc[0, "FE_YearStart"] = (
    "not-a-year"
)

test_results = []

for case_name, frame in test_cases.items():
    start = time.perf_counter()

    try:
        predictions = yield_model.predict(frame)
        elapsed_ms = (time.perf_counter() - start) * 1000

        predictions = np.asarray(predictions)
        test_results.append({
            "test_case": case_name,
            "status": "PASS",
            "prediction_count": len(predictions),
            "all_finite": bool(np.isfinite(predictions).all()),
            "elapsed_ms": round(elapsed_ms, 3),
            "error": None,
        })

        print(f"\n{case_name}: PASS")
        print("Predictions:", predictions)
        print("All finite :", bool(np.isfinite(predictions).all()))
        print("Elapsed ms :", round(elapsed_ms, 3))

    except Exception as exc:
        elapsed_ms = (time.perf_counter() - start) * 1000
        test_results.append({
            "test_case": case_name,
            "status": "REJECTED_OR_FAILED",
            "prediction_count": None,
            "all_finite": None,
            "elapsed_ms": round(elapsed_ms, 3),
            "error": f"{type(exc).__name__}: {exc}",
        })

        print(f"\n{case_name}: REJECTED_OR_FAILED")
        print("Error:", f"{type(exc).__name__}: {exc}")

test_results_df = pd.DataFrame(test_results)

print("\nTEST SUMMARY")
display(test_results_df)


C:\Users\gsr58\AppData\Local\Temp\ipykernel_10632\255822102.py:42: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'not-a-year' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  test_cases["invalid_year_string"].loc[0, "FE_YearStart"] = (



valid_inputs: PASS
Predictions: [4229.91783273 4671.21779049]
All finite : True
Elapsed ms : 2093.572

unseen_categories: PASS
Predictions: [1414.3089463  4671.21779049]
All finite : True
Elapsed ms : 13.441

missing_crop: PASS
Predictions: [1439.16977322 4671.21779049]
All finite : True
Elapsed ms : 12.775

missing_year: PASS
Predictions: [4144.13051873 4671.21779049]
All finite : True
Elapsed ms : 13.123

invalid_year_string: REJECTED_OR_FAILED
Error: ValueError: could not convert string to float: 'not-a-year'

TEST SUMMARY


,test_case,status,prediction_count,all_finite,elapsed_ms,error
0,valid_inputs,PASS,2.0,True,2093.572,None
1,unseen_categories,PASS,2.0,True,13.441,None
2,missing_crop,PASS,2.0,True,12.775,None
3,missing_year,PASS,2.0,True,13.123,None
4,invalid_year_string,REJECTED_OR_FAILED,NaN,None,3.943,ValueError: could not convert string to float:...


In [13]:

# Repeat inference with the same valid inputs.
predictions_run_1 = np.asarray(yield_model.predict(test_inputs))
predictions_run_2 = np.asarray(yield_model.predict(test_inputs))
predictions_run_3 = np.asarray(yield_model.predict(test_inputs))

deterministic = (
    np.allclose(predictions_run_1, predictions_run_2, rtol=0, atol=1e-10)
    and np.allclose(predictions_run_1, predictions_run_3, rtol=0, atol=1e-10)
)

finite_outputs = bool(np.isfinite(predictions_run_1).all())
numeric_outputs = bool(np.issubdtype(predictions_run_1.dtype, np.number))
nonnegative_outputs = bool((predictions_run_1 >= 0).all())

prediction_diagnostics = pd.DataFrame({
    "Crop": test_inputs["Crop"],
    "Season": test_inputs["Season"],
    "Year": test_inputs["FE_YearStart"],
    "Predicted_Yield": predictions_run_1,
})

print("DETERMINISM AND OUTPUT CHECKS")
print("-" * 60)
print("Deterministic repeated predictions:", deterministic)
print("Numeric outputs                  :", numeric_outputs)
print("All outputs finite               :", finite_outputs)
print("All outputs nonnegative          :", nonnegative_outputs)

print("\nPREDICTION DIAGNOSTICS")
display(prediction_diagnostics)

assert deterministic, "Repeated predictions differ."
assert numeric_outputs, "Predictions are not numeric."
assert finite_outputs, "Predictions contain NaN or infinity."

print(
    "\nCore output checks passed. "
    "Nonnegativity is recorded as a diagnostic, not a universal "
    "domain-validity guarantee."
)


DETERMINISM AND OUTPUT CHECKS
------------------------------------------------------------
Deterministic repeated predictions: True
Numeric outputs                  : True
All outputs finite               : True
All outputs nonnegative          : True

PREDICTION DIAGNOSTICS


,Crop,Season,Year,Predicted_Yield
0,Rice,Kharif,2023,4229.917833
1,Wheat,Rabi,2024,4671.217790



Core output checks passed. Nonnegativity is recorded as a diagnostic, not a universal domain-validity guarantee.


In [14]:

from datetime import datetime, timezone

# Summarize observed results and known deployment gaps.
readiness_checks = {
    "model_artifact_loaded": True,
    "metadata_loaded": True,
    "metadata_contract_matches_pipeline": all(checks.values()),
    "valid_sample_predictions_succeeded": (
        test_results_df.loc[
            test_results_df["test_case"] == "valid_inputs", "status"
        ].iloc[0] == "PASS"
    ),
    "unseen_categories_accepted": (
        test_results_df.loc[
            test_results_df["test_case"] == "unseen_categories", "status"
        ].iloc[0] == "PASS"
    ),
    "missing_crop_rejected": (
        test_results_df.loc[
            test_results_df["test_case"] == "missing_crop", "status"
        ].iloc[0] != "PASS"
    ),
    "missing_year_rejected": (
        test_results_df.loc[
            test_results_df["test_case"] == "missing_year", "status"
        ].iloc[0] != "PASS"
    ),
    "malformed_year_rejected": (
        test_results_df.loc[
            test_results_df["test_case"] == "invalid_year_string", "status"
        ].iloc[0] != "PASS"
    ),
    "repeat_predictions_deterministic": bool(deterministic),
    "predictions_numeric_and_finite": bool(numeric_outputs and finite_outputs),
    "sample_predictions_nonnegative": bool(nonnegative_outputs),
}

known_gaps = [
    "Missing Crop and FE_YearStart values currently produce predictions; "
    "the serving layer should reject incomplete requests.",
    "Unseen crop, state, and district values are accepted, but prediction "
    "reliability for unseen categories is not established.",
    "Only two synthetic input rows were used for this smoke test.",
    "No evidence-based agronomic output bounds have been established.",
    "The initial latency measurement is not a representative benchmark.",
    "API behavior, authentication, monitoring, logging, load testing, "
    "deployment security, rollback, and service-level objectives remain "
    "outside this notebook's tested scope.",
    "Notebook 16 records that the final holdout was repeatedly inspected; "
    "its metrics should not be presented as an untouched final test.",
]

if all([
    readiness_checks["model_artifact_loaded"],
    readiness_checks["metadata_loaded"],
    readiness_checks["metadata_contract_matches_pipeline"],
    readiness_checks["valid_sample_predictions_succeeded"],
    readiness_checks["repeat_predictions_deterministic"],
    readiness_checks["predictions_numeric_and_finite"],
]) and not any([
    readiness_checks["missing_crop_rejected"],
    readiness_checks["missing_year_rejected"],
]):
    verdict = "CONDITIONALLY READY FOR INTEGRATION"
else:
    verdict = "NOT READY FOR INTEGRATION"

readiness_report = {
    "report_name": "yield_model_deployment_readiness",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_path": str(model_path),
    "metadata_path": str(metadata_path),
    "model_name": model_metadata.get("model_name"),
    "features": metadata_features,
    "target": metadata_target,
    "checks": readiness_checks,
    "known_gaps": known_gaps,
    "verdict": verdict,
    "interpretation": (
        "This report assesses notebook-level artifact and inference checks. "
        "It is not a certification of production readiness."
    ),
}

report_path = model_dir / "yield_model_deployment_readiness_report.json"

with report_path.open("w", encoding="utf-8") as file:
    json.dump(readiness_report, file, indent=2, ensure_ascii=False)

print("READINESS VERDICT:", verdict)
print("\nCHECK RESULTS")
for name, passed in readiness_checks.items():
    print(f"{'PASS' if passed else 'GAP '} | {name}")

print("\nKNOWN GAPS")
for index, gap in enumerate(known_gaps, start=1):
    print(f"{index}. {gap}")

print("\nREPORT SAVED TO:", report_path)


READINESS VERDICT: CONDITIONALLY READY FOR INTEGRATION

CHECK RESULTS
PASS | model_artifact_loaded
PASS | metadata_loaded
PASS | metadata_contract_matches_pipeline
PASS | valid_sample_predictions_succeeded
PASS | unseen_categories_accepted
GAP  | missing_crop_rejected
GAP  | missing_year_rejected
PASS | malformed_year_rejected
PASS | repeat_predictions_deterministic
PASS | predictions_numeric_and_finite
PASS | sample_predictions_nonnegative

KNOWN GAPS
1. Missing Crop and FE_YearStart values currently produce predictions; the serving layer should reject incomplete requests.
2. Unseen crop, state, and district values are accepted, but prediction reliability for unseen categories is not established.
3. Only two synthetic input rows were used for this smoke test.
4. No evidence-based agronomic output bounds have been established.
5. The initial latency measurement is not a representative benchmark.
6. API behavior, authentication, monitoring, logging, load testing, deployment security, ro

In [15]:

# Final report correction: distinguish technical acceptance from reliability.

readiness_report["checks"]["unseen_categories_accepted"] = True
readiness_report["checks"]["unseen_category_reliability_verified"] = False
readiness_report["checks"]["missing_crop_rejected"] = False
readiness_report["checks"]["missing_year_rejected"] = False

readiness_report["known_gaps"] = list(dict.fromkeys(
    readiness_report["known_gaps"] + [
        "Unseen-category acceptance is a technical behavior, not evidence "
        "of reliable predictions for unseen categories."
    ]
))

readiness_report["verdict"] = "CONDITIONALLY READY FOR INTEGRATION"
readiness_report["interpretation"] = (
    "The saved model passes basic artifact and inference smoke tests. "
    "Integration must include required-field validation and explicit "
    "handling of unseen categories. Production readiness has not been "
    "established."
)

with report_path.open("w", encoding="utf-8") as file:
    json.dump(readiness_report, file, indent=2, ensure_ascii=False)

print("Updated report:", report_path)
print("Final verdict:", readiness_report["verdict"])
print("Missing required-input safeguards: outstanding")
print("Unseen-category reliability: unverified")
print("Production readiness: not established")


Updated report: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\yield_prediction\yield_model_deployment_readiness_report.json
Final verdict: CONDITIONALLY READY FOR INTEGRATION
Missing required-input safeguards: outstanding
Unseen-category reliability: unverified
Production readiness: not established


## Final Conclusion — Yield Model Deployment Readiness

### 1. Objective
Notebook 17 assessed whether the selected yield prediction model and its saved artifacts are suitable for integration into the BharatCrop Intelligence application.

### 2. Artifacts Verified
- **Selected model:** `individual_crop_hgb_location.joblib`
- **Model metadata:** `individual_crop_hgb_location_metadata.json`
- **Readiness report:** `yield_model_deployment_readiness_report.json`

### 3. Key Findings
- The saved model loads and produces finite, deterministic predictions for the tested inputs.
- The model accepts previously unseen categorical values through its configured preprocessing pipeline; prediction reliability for unseen categories has not been established.
- Missing required values were not consistently rejected by the raw model pipeline, demonstrating the need for a separate input-validation layer.
- Prediction latency has not been benchmarked on a representative workload.
- Production concerns such as API security, monitoring, logging, load testing, rollback, and service-level objectives remain unverified.
- The 2023–2024 holdout data was inspected during model development and diagnostics; it must not be described as a pristine final test set.

### 4. Final Decision
**CONDITIONALLY READY FOR INTEGRATION**

The selected model can proceed to the application-integration stage, provided that a separate inference interface validates inputs and outputs. This decision does not establish production readiness, generalization to unseen locations, or agronomic reliability.

### 5. Next Implementation Stage
The next stage will implement a standalone Python inference module under `src/yield_prediction/`, with automated tests under `tests/`. This module will load the saved model, enforce the input contract, validate predictions, and provide single-record and batch prediction interfaces.

The inference module is application code, **not another notebook**. It will reuse the existing saved model and metadata; retraining is not part of this stage.

### 6. Notebook Status
Notebook 17 is complete and should be frozen. Further validation of the inference interface, API integration, and production safeguards will be performed in the application code and its automated tests, rather than by extending this notebook.